In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
from Leiden_analyze import Leidenanalyzer
from Azimuth_celltype import AZIMUTHvisualizer # celltype直接點在umap
import scanpy as sc
import pandas as pd

/staging/biology/jane0528/miniconda3/envs/scrublet_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#載入Harmony+Azimuth後的Adata
adata_merged=sc.read_h5ad('/staging/biology/jane0528/NMOSD/scRNA/Dataset/My_merged_protein_coding_genes/Adata/My_merged_Azimuth(protein_coding).h5ad')
#載入我的已分群Adata
adata_clustered=sc.read_h5ad('/staging/biology/jane0528/NMOSD/scRNA/Dataset/My_merged_protein_coding_genes/Adata/Leiden_all_results(Harmony_nn_25_res_0.6).h5ad')

In [3]:
import numpy as np
adata_merged.obs["Condition"] = np.where(
    adata_merged.obs["orig.ident"].astype(str).str.startswith("nmo"),
    "NMOSD",
    "Control"
)

In [4]:
adata_predicted=adata_merged.copy()

In [5]:
#parameters
cluster_key = "leiden_nn25_res0.6" #原始分群label
n_neighbors=25
ndim_pca=50

#把"predicted_label"加入leiden分群好的adata中
adata_predicted.obs[cluster_key]=adata_clustered.obs[cluster_key]

In [6]:
celltype1_df=pd.read_csv("/staging/biology/jane0528/NMOSD/scRNA/Dataset/My_merged_protein_coding_genes_V2/Harmony_Azimuth_celltype_composition/Harmony50/leiden_nn25_res0.6_predicted.celltype.l1.csv")
celltype1_df_v1=celltype1_df.loc[:,['leiden_nn25_res0.6','Dominant_celltype']]
celltype1_df_v1.rename(columns={"Dominant_celltype":"Major_celltype"}, inplace=True)

In [7]:
obs = adata_predicted.obs.copy()
obs["cell_id"] = obs.index  # 存 row names

obs["leiden_nn25_res0.6"] = obs["leiden_nn25_res0.6"].astype(int)
celltype1_df_v1["leiden_nn25_res0.6"] = celltype1_df_v1["leiden_nn25_res0.6"].astype(int)

obs = obs.merge(celltype1_df_v1, on="leiden_nn25_res0.6", how="left")
obs = obs.set_index("cell_id")  # 還原 row names

adata_predicted.obs = obs

In [8]:
adata_predicted.obs

,orig.ident,nCount_RNA,nFeature_RNA,percent.mt,predicted.celltype.l1.score,predicted.celltype.l1,predicted.celltype.l2.score,predicted.celltype.l2,predicted.celltype.l3.score,predicted.celltype.l3,mapping.score,Condition,leiden_nn25_res0.6,Major_celltype
cell_id,,,,,,,,,,,,,,
nmo008_ACACTGACAATGAAAC-1,nmo008,11860.0,2988,3.971332,0.685193,CD4 T,0.624930,CD4 TCM,0.479567,CD4 TCM_1,0.160075,NMOSD,3,CD4 T
nmo008_AAGCCGCGTCGAAAGC-1,nmo008,11752.0,2838,3.556841,1.000000,CD4 T,1.000000,CD4 TCM,0.987932,CD4 TCM_2,0.821969,NMOSD,3,CD4 T
nmo008_TCAGATGGTCAACTGT-1,nmo008,11642.0,3070,3.848136,1.000000,CD4 T,1.000000,CD4 TCM,0.992778,CD4 TCM_2,0.949871,NMOSD,3,CD4 T
nmo008_CTAACTTGTTAAAGTG-1,nmo008,10934.0,3116,5.341138,1.000000,CD4 T,1.000000,CD4 TCM,1.000000,CD4 TCM_2,0.907426,NMOSD,3,CD4 T
nmo008_AGAGCTTAGAGTAAGG-1,nmo008,10910.0,3136,6.186984,0.852914,CD8 T,0.852914,CD8 TEM,0.469027,CD8 TEM_1,0.324083,NMOSD,1,CD8 T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Control3_TTTGTCACACGGACAA-1,Control3,4684.0,1876,2.625961,0.961912,other T,0.898177,MAIT,0.898177,MAIT,0.824499,Control,9,other T
Control3_TTTGTCACAGCAGTTT-1,Control3,2438.0,1224,1.886792,0.902620,CD8 T,0.886428,CD8 TEM,0.758866,CD8 TEM_2,0.558437,Control,1,CD8 T
Control3_TTTGTCAGTCTTCAAG-1,Control3,2902.0,1424,1.654032,1.000000,NK,1.000000,NK,1.000000,NK_2,0.811304,Control,2,NK


In [9]:
from pathlib import Path
import pandas as pd

base_dir = Path("/staging/biology/jane0528/NMOSD/scRNA/Dataset/My_merged_Celltype2_Reclustered/Azimuth_subcelltype_result")

# 讀所有 *_subtype.csv
csv_paths = sorted(base_dir.glob("*_subtype_*.csv"))
print("Found:", [p.name for p in csv_paths])

dfs = []
for p in csv_paths:
    df = pd.read_csv(p)  # columns: cell_id, cluster_annotation
    df = df.rename(columns={"cluster_annotation": "sub_celltype"})
    df["source"] = p.stem  # 例如 B_subtype, T_subtype
    dfs.append(df)

map_df = pd.concat(dfs, ignore_index=True)

# 檢查是否有出現同一個 cell_id
dup = map_df["cell_id"].duplicated().sum()
print("Duplicated cell_id:", dup)

map_df.head()


Found: ['B_subtype_nn30.csv', 'Mono_DC_subtype_nn30.csv', 'NK_subtype_nn30.csv', 'T_subtype_nn30.csv']
Duplicated cell_id: 0


,cell_id,sub_celltype,source
0,nmo008_AGAATAGCAGGTCTCG-1,B memory,B_subtype_nn30
1,nmo008_GCTCTGTGTACCGTTA-1,B intermediate,B_subtype_nn30
2,nmo008_ATAACGCCAATTCCTT-1,B naive,B_subtype_nn30
3,nmo008_TGAGAGGCATGACATC-1,B naive,B_subtype_nn30
4,nmo008_GAGGTGAGTATTAGCC-1,B memory,B_subtype_nn30


In [10]:
# 如果你確定 cell_id 唯一：
sub_map = map_df.set_index("cell_id")["sub_celltype"].to_dict()

# 確保 obs 有 cell_id 欄（沒有就補）
if "cell_id" not in adata_predicted.obs.columns:
    adata_predicted.obs["cell_id"] = adata_predicted.obs.index

adata_predicted.obs["sub_celltype"] = adata_predicted.obs["cell_id"].map(sub_map)

# 沒被標到的（例如不是 B/T/NK/MonoDC 的）會是 NaN，你可以補 Unknown
#adata_predicted.obs["sub_celltype"] = adata_predicted.obs["sub_celltype"].fillna("Unknown").astype("category")


In [11]:
adata_predicted.obs

,orig.ident,nCount_RNA,nFeature_RNA,percent.mt,predicted.celltype.l1.score,predicted.celltype.l1,predicted.celltype.l2.score,predicted.celltype.l2,predicted.celltype.l3.score,predicted.celltype.l3,mapping.score,Condition,leiden_nn25_res0.6,Major_celltype,cell_id,sub_celltype
cell_id,,,,,,,,,,,,,,,,
nmo008_ACACTGACAATGAAAC-1,nmo008,11860.0,2988,3.971332,0.685193,CD4 T,0.624930,CD4 TCM,0.479567,CD4 TCM_1,0.160075,NMOSD,3,CD4 T,nmo008_ACACTGACAATGAAAC-1,CD4 TCM
nmo008_AAGCCGCGTCGAAAGC-1,nmo008,11752.0,2838,3.556841,1.000000,CD4 T,1.000000,CD4 TCM,0.987932,CD4 TCM_2,0.821969,NMOSD,3,CD4 T,nmo008_AAGCCGCGTCGAAAGC-1,CD4 TCM
nmo008_TCAGATGGTCAACTGT-1,nmo008,11642.0,3070,3.848136,1.000000,CD4 T,1.000000,CD4 TCM,0.992778,CD4 TCM_2,0.949871,NMOSD,3,CD4 T,nmo008_TCAGATGGTCAACTGT-1,CD4 TCM
nmo008_CTAACTTGTTAAAGTG-1,nmo008,10934.0,3116,5.341138,1.000000,CD4 T,1.000000,CD4 TCM,1.000000,CD4 TCM_2,0.907426,NMOSD,3,CD4 T,nmo008_CTAACTTGTTAAAGTG-1,CD4 TCM
nmo008_AGAGCTTAGAGTAAGG-1,nmo008,10910.0,3136,6.186984,0.852914,CD8 T,0.852914,CD8 TEM,0.469027,CD8 TEM_1,0.324083,NMOSD,1,CD8 T,nmo008_AGAGCTTAGAGTAAGG-1,CD8 TEM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Control3_TTTGTCACACGGACAA-1,Control3,4684.0,1876,2.625961,0.961912,other T,0.898177,MAIT,0.898177,MAIT,0.824499,Control,9,other T,Control3_TTTGTCACACGGACAA-1,MAIT
Control3_TTTGTCACAGCAGTTT-1,Control3,2438.0,1224,1.886792,0.902620,CD8 T,0.886428,CD8 TEM,0.758866,CD8 TEM_2,0.558437,Control,1,CD8 T,Control3_TTTGTCACAGCAGTTT-1,CD8 TEM
Control3_TTTGTCAGTCTTCAAG-1,Control3,2902.0,1424,1.654032,1.000000,NK,1.000000,NK,1.000000,NK_2,0.811304,Control,2,NK,Control3_TTTGTCAGTCTTCAAG-1,NK


In [12]:
# 檢查是否有nan
adata_predicted.obs["sub_celltype"].isna().sum()

np.int64(0)

In [13]:
adata_predicted.obs.drop(columns={"cell_id"},inplace=True)

In [14]:
adata_predicted.obs

,orig.ident,nCount_RNA,nFeature_RNA,percent.mt,predicted.celltype.l1.score,predicted.celltype.l1,predicted.celltype.l2.score,predicted.celltype.l2,predicted.celltype.l3.score,predicted.celltype.l3,mapping.score,Condition,leiden_nn25_res0.6,Major_celltype,sub_celltype
cell_id,,,,,,,,,,,,,,,
nmo008_ACACTGACAATGAAAC-1,nmo008,11860.0,2988,3.971332,0.685193,CD4 T,0.624930,CD4 TCM,0.479567,CD4 TCM_1,0.160075,NMOSD,3,CD4 T,CD4 TCM
nmo008_AAGCCGCGTCGAAAGC-1,nmo008,11752.0,2838,3.556841,1.000000,CD4 T,1.000000,CD4 TCM,0.987932,CD4 TCM_2,0.821969,NMOSD,3,CD4 T,CD4 TCM
nmo008_TCAGATGGTCAACTGT-1,nmo008,11642.0,3070,3.848136,1.000000,CD4 T,1.000000,CD4 TCM,0.992778,CD4 TCM_2,0.949871,NMOSD,3,CD4 T,CD4 TCM
nmo008_CTAACTTGTTAAAGTG-1,nmo008,10934.0,3116,5.341138,1.000000,CD4 T,1.000000,CD4 TCM,1.000000,CD4 TCM_2,0.907426,NMOSD,3,CD4 T,CD4 TCM
nmo008_AGAGCTTAGAGTAAGG-1,nmo008,10910.0,3136,6.186984,0.852914,CD8 T,0.852914,CD8 TEM,0.469027,CD8 TEM_1,0.324083,NMOSD,1,CD8 T,CD8 TEM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Control3_TTTGTCACACGGACAA-1,Control3,4684.0,1876,2.625961,0.961912,other T,0.898177,MAIT,0.898177,MAIT,0.824499,Control,9,other T,MAIT
Control3_TTTGTCACAGCAGTTT-1,Control3,2438.0,1224,1.886792,0.902620,CD8 T,0.886428,CD8 TEM,0.758866,CD8 TEM_2,0.558437,Control,1,CD8 T,CD8 TEM
Control3_TTTGTCAGTCTTCAAG-1,Control3,2902.0,1424,1.654032,1.000000,NK,1.000000,NK,1.000000,NK_2,0.811304,Control,2,NK,NK


In [15]:
adata_predicted.write_h5ad("/staging/biology/jane0528/NMOSD/scRNA/Dataset/My_merged_Celltype2_Reclustered/Adata/Harmony_Azimuth_Major_Minor_celltype.h5ad")

In [16]:
#parameters
cluster_key = "sub_celltype" #原始分群label
n_neighbors=25
ndim_pca=50

In [17]:
viz = AZIMUTHvisualizer(adata_predicted,use_rep='X_harmony',ndim_pca=ndim_pca, base_output="../Dataset/My_merged_Celltype2_Reclustered")
viz.prepare_umap(n_neighbors=n_neighbors)
viz.plot_umap_and_composition(cluster_key)

UMAP computed using 25 neighbors and 50 PCs.
All plots saved to: ../Dataset/My_merged_Celltype2_Reclustered/Azimuth_plot/Harmony50/sub_celltype


AnnData object with n_obs × n_vars = 218212 × 16489
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'predicted.celltype.l1.score', 'predicted.celltype.l1', 'predicted.celltype.l2.score', 'predicted.celltype.l2', 'predicted.celltype.l3.score', 'predicted.celltype.l3', 'mapping.score', 'Condition', 'leiden_nn25_res0.6', 'Major_celltype', 'sub_celltype', 'cluster_annotation'
    var: 'vf_vst_counts_mean', 'vf_vst_counts_variance', 'vf_vst_counts_variance.expected', 'vf_vst_counts_variance.standardized', 'vf_vst_counts_variable', 'vf_vst_counts_rank', 'var.features', 'var.features.rank'
    uns: 'neighbors', 'umap', 'cluster_annotation_colors', 'orig.ident_colors'
    obsm: 'X_harmony', 'X_integrated_dr', 'X_pca', 'X_ref.umap', 'azimuth_harmony', 'X_umap'
    varm: 'HARMONY', 'PCs'
    obsp: 'distances', 'connectivities'